# Lab 4.2 — GraphRAG on HotpotQA
**Module IV · LLMs + GNNs Together**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_solutions/blob/main/module-4-hybrid/lab4_2_graphrag.ipynb)

---

## What you will do
1. Load **HotpotQA** (Yang et al., EMNLP 2018) — the standard benchmark for multi-hop question answering. Every question in HotpotQA requires reasoning over **exactly two supporting paragraphs** simultaneously.
2. Evaluate **basic RAG** (dense retrieval) on HotpotQA and measure how often it retrieves both supporting paragraphs.
3. Build a **paragraph similarity graph** over each question's candidates and implement **GraphRAG**: seed retrieval → graph expansion → re-ranking.
4. Compare RAG vs GraphRAG across all questions and separately on **bridge** vs **comparison** question types, where the benefit of graph structure differs.
5. `[Extension]` Plot retrieval recall@k and analyse where graph expansion helps most.

## Why HotpotQA and not a toy dataset?
HotpotQA is used in the retrieval-augmented generation literature as the canonical multi-hop benchmark. Our previous TechRetail KB was too small (8 documents, hand-crafted questions) to meaningfully distinguish retrieval strategies. HotpotQA provides:
- **113k real questions** from Wikipedia, validated by humans
- **Annotated supporting facts** — ground truth for which paragraphs are needed
- **Two question types**: bridge (chain of reasoning) and comparison (parallel look-up)
- A **distractor setting** where each question has 10 candidate paragraphs: 2 supporting + 8 distractors
- Results comparable to published RAG and GraphRAG baselines

## Evaluation metric
**Full Retrieval Recall@k**: fraction of questions for which the top-k retrieved paragraphs include *both* supporting paragraphs. This directly measures whether the retrieval system provides the LLM with everything it needs.

## Prerequisites
Labs 2.3 (RAG) and 3.1 (NetworkX) completed.

**Estimated time:** 60–75 min

---
## 0 · Setup

In [ ]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Cloning the course solutions repository and installing dependencies...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/DanielFPerez/llm-gnns-course_solutions.git"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         "llm-gnns-course_solutions/environment/requirements.txt"], check=True)
    sys.path.insert(0, "llm-gnns-course_solutions")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict

from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from utils import SimpleLLM

st_model = SentenceTransformer("all-MiniLM-L6-v2")
llm = SimpleLLM()

print(f"Sentence-transformer loaded (dim={st_model.get_sentence_embedding_dimension()})")
print(f"LLM: {llm}")

---
## 1 · HotpotQA: the multi-hop QA benchmark

**HotpotQA** (Yang et al., EMNLP 2018) contains questions that require reasoning over two Wikipedia paragraphs. The **distractor setting** we use here gives each question:
- 2 **supporting paragraphs** (the paragraphs you need to answer the question)
- 8 **distractor paragraphs** (plausible-looking but not needed)

There are two question types:
- **Bridge**: you must follow an entity chain — find the bridge entity in paragraph 1, then look up its attribute in paragraph 2.
  - *"What nationality is the director of the film that won the 1972 Academy Award for Best Picture?"*
- **Comparison**: look up the same attribute for two entities and compare.
  - *"Were Scott Derrickson and Ed Wood both American directors?"*

Bridge questions are harder for standard retrieval because the two supporting paragraphs are connected by an intermediate entity that may not be explicit in the question.

In [ ]:
# Load the validation split of HotpotQA (distractor setting)
# Downloads ~120 MB on first run, then cached at ~/.cache/huggingface/datasets/
print("Loading HotpotQA (distractor)...")
hotpot_val = load_dataset("hotpot_qa", "distractor",
                          split="validation", trust_remote_code=True)
print(f"Validation examples: {len(hotpot_val):,}")

# Inspect one example
ex = hotpot_val[0]
print(f"\nExample question  : {ex['question']}")
print(f"Answer            : {ex['answer']}")
print(f"Type              : {ex['type']}")
print(f"Level             : {ex['level']}")
print(f"Supporting titles : {ex['supporting_facts']['title']}")
print(f"\n10 candidate paragraphs:")
for title in ex['context']['title']:
    marker = "✓" if title in ex['supporting_facts']['title'] else "✗"
    print(f"  {marker} {title}")

### Exercise 4.2.1 `[Core]` — Dataset exploration

Using the full 7,405-example validation set:
1. How many bridge questions and how many comparison questions are there?
2. What is the distribution of difficulty levels (easy / medium / hard)?
3. What fraction of supporting paragraphs appear as the **first** candidate (index 0) vs. elsewhere? This tells us whether position bias could inflate retrieval scores.

In [ ]:
# YOUR CODE HERE
# Hint: Using the full 7,405-example validation set:
raise NotImplementedError("Complete this exercise")

---
## 2 · Embedding paragraphs and building the evaluation corpus

We work with a **300-question evaluation subset** — large enough for reliable statistics but fast to embed on CPU. For each question we embed all 10 candidate paragraphs and tag whether each is a supporting fact.

The setup mirrors a realistic retrieval scenario: given 10 candidate paragraphs and a question, rank them so that both supporting paragraphs appear in the top-2 (or top-3) retrieved results.

In [ ]:
# Helper functions

def parse_example(ex: dict) -> dict:
    """Convert a raw HotpotQA example into a clean working dict."""
    titles    = ex["context"]["title"]
    sentences = ex["context"]["sentences"]
    sup_titles = set(ex["supporting_facts"]["title"])

    paragraphs = [
        {
            "title": title,
            "text":  " ".join(sents),
            "is_supporting": title in sup_titles,
        }
        for title, sents in zip(titles, sentences)
    ]
    return {
        "id":         ex["id"],
        "question":   ex["question"],
        "answer":     ex["answer"],
        "type":       ex["type"],
        "level":      ex["level"],
        "paragraphs": paragraphs,   # list of 10 dicts
    }


def full_recall_at_k(examples: list[dict], retrieve_fn, k: int = 2) -> float:
    """
    Fraction of questions for which top-k retrieved paragraphs include
    ALL supporting paragraphs (full recall).
    """
    correct = 0
    for ex in examples:
        retrieved = retrieve_fn(ex)
        retrieved_titles = {p["title"] for p in retrieved[:k]}
        sup_titles = {p["title"] for p in ex["paragraphs"] if p["is_supporting"]}
        if sup_titles.issubset(retrieved_titles):
            correct += 1
    return correct / len(examples)


print("Helper functions defined.")

### Exercise 4.2.2 `[Core]` — Embed the evaluation subset

1. Take the first 300 examples from `hotpot_val` and parse them with `parse_example`.
2. Collect all 3,000 paragraph texts (300 questions × 10 paragraphs) and embed them in one batch with `st_model.encode`.
3. L2-normalise the embeddings (so cosine similarity = dot product).
4. Store each paragraph's embedding back into the parsed dict under key `"embedding"`.
5. Print how many bridge vs comparison examples are in the subset.

In [ ]:
# YOUR CODE HERE
# Hint: 1. Take the first 300 examples from hotpot_val and parse them with parse_example.
raise NotImplementedError("Complete this exercise")

---
## 3 · Basic RAG: dense retrieval baseline

For each question we embed it with the sentence-transformer and rank the 10 candidate paragraphs by cosine similarity. This is the simplest possible dense retrieval baseline — equivalent to what `faiss.IndexFlatIP` does, but over only 10 candidates so we compute it directly.

In [ ]:
# Embed all questions (300) for fast retrieval
question_embs = st_model.encode(
    [ex["question"] for ex in examples],
    batch_size=64, show_progress_bar=True,
    convert_to_numpy=True,
).astype(np.float32)
question_embs /= np.linalg.norm(question_embs, axis=1, keepdims=True)

for i, ex in enumerate(examples):
    ex["question_embedding"] = question_embs[i]

print("Question embeddings computed.")

### Exercise 4.2.3 `[Core]` — Implement and evaluate basic RAG

1. Implement `rag_retrieve(ex)` that ranks the 10 paragraphs by cosine similarity to the question and returns them sorted (highest first).
2. Compute **Full Retrieval Recall@2** and **@3** on all 300 examples.
   - Recall@2: did we get both supporting paragraphs in top-2?
   - Recall@3: slack version (some retrieval papers use @3 as the budget)
3. Compute recall separately for bridge and comparison questions.

In [ ]:
# YOUR CODE HERE
# Hint: 1. Implement rag_retrieve(ex) that ranks the 10 paragraphs by cosine similarity to the question and returns them sort...

def rag_retrieve(ex: dict) -> list[dict]:
    pass  # replace this

> **Typical results:** RAG achieves ~50–60% Full Recall@2 on HotpotQA distractor. For comparison questions the two supporting paragraphs are independently relevant to the question, so plain cosine similarity often surfaces both. For bridge questions, the second supporting paragraph may be about an *intermediate entity* that is not mentioned in the question at all — cosine similarity to the question embedding is low, and retrieval fails.
>
> Published dense retrieval numbers on HotpotQA distractor for `all-MiniLM-L6-v2`: ~55% Full Recall@2. State-of-the-art systems (e.g., MDR, Baleen) exceed 90% by chaining retrieval steps. GraphRAG is a single-step way to improve over plain dense retrieval.

---
## 4 · Building a paragraph similarity graph

For each question, we build a small 10-node graph where nodes are candidate paragraphs and edges connect paragraphs that are semantically similar. The intuition for bridge questions:

```
Question: "Who directed the film that starred [Actor X]?"

Para 1 (supporting): "[Film Y] starred [Actor X]. It was directed by [Director Z]."
Para 2 (supporting): "[Director Z] was born in..."   ← may not match query directly
Distractor paras:    About other films, other actors.
```

Para 1 is highly similar to the question (shares "Actor X"). Para 2 mentions the same director as Para 1, so **Para 1 and Para 2 share an entity and are semantically connected**. If RAG retrieves Para 1 as a seed, one graph hop reaches Para 2 — even though Para 2 alone would not rank high by query similarity.

### Exercise 4.2.4 `[Core]` — Build the per-question paragraph graph

Implement `build_para_graph(ex, threshold=0.5)` that:
1. Computes the 10×10 cosine similarity matrix between all paragraph embeddings.
2. Adds an edge between paragraphs $i$ and $j$ (where $i \neq j$) if their cosine similarity exceeds `threshold`.
3. Returns an `nx.Graph` with node attribute `"is_supporting"`.

Then apply it to the first 5 bridge examples and print the average edge count and the fraction of supporting–supporting edges. Are the two supporting paragraphs directly connected more often than chance?

In [ ]:
# YOUR CODE HERE
# Hint: Implement build_para_graph(ex, threshold=0.5) that:

def build_para_graph(ex: dict, threshold: float = 0.5) -> nx.Graph:
    pass  # replace this

---
## 5 · GraphRAG: seed retrieval + graph expansion + re-ranking

The GraphRAG algorithm for each question:

1. **Seed**: retrieve top-`k_seed` paragraphs by cosine similarity with the question.
2. **Expand**: add all 1-hop neighbours of the seed nodes in the paragraph graph.
3. **Re-rank**: among the expanded set, sort by cosine similarity with the question.
4. **Return** top-`m` paragraphs.

### Exercise 4.2.5 `[Core]` — Implement GraphRAG retrieval

Implement `graphrag_retrieve(ex, k_seed=1, hops=1, threshold=0.5)` using `build_para_graph` and the question embeddings already stored in `ex`.

**Design choice — why `k_seed=1`?** With only 10 candidates, seeding with the single most similar paragraph and expanding 1-hop gives the graph maximum room to discover the second supporting paragraph. With `k_seed=2` we are already close to just doing RAG@2.

In [ ]:
# YOUR CODE HERE
# Hint: Implement graphrag_retrieve(ex, k_seed=1, hops=1, threshold=0.5) using build_para_graph and the question embeddings a...

def graphrag_retrieve(ex: dict, k_seed: int = 1,
    pass  # replace this

### Exercise 4.2.6 `[Core]` — Evaluate GraphRAG and compare with RAG

1. Compute Full Retrieval Recall@2 and @3 for GraphRAG on all 300 examples, and separately for bridge and comparison questions.
2. Plot a grouped bar chart comparing RAG vs GraphRAG on all three subsets (all / bridge / comparison).
3. Is the improvement larger for bridge or comparison questions? Why?

In [ ]:
# YOUR CODE HERE
# Hint: 1. Compute Full Retrieval Recall@2 and @3 for GraphRAG on all 300 examples, and separately for bridge and comparison ...
raise NotImplementedError("Complete this exercise")

> **Key finding:** GraphRAG's gain is concentrated on **bridge questions**. For bridge questions, the seed paragraph (directly similar to the question) is connected via the graph to the second supporting paragraph (which mentions the bridge entity). For comparison questions, both supporting paragraphs are independently relevant to the question — plain cosine similarity already retrieves both, so the graph adds little.
>
> This confirms the theoretical motivation: graph-augmented retrieval is most valuable when questions require **chaining through intermediate entities** — exactly the setting where LLMs also hallucinate most readily.

---
## 6 · End-to-end: does better retrieval yield better answers?

Better retrieval recall should translate to better answers. We test on a small sample (20 questions) to keep inference time manageable, and measure whether the ground-truth answer appears in the generated text.

In [ ]:
import re

def normalize(s: str) -> str:
    """Lowercase and strip punctuation for loose answer matching."""
    return re.sub(r"[^a-z0-9 ]", " ", s.lower()).strip()


def answer_correct(response: str, gold: str) -> bool:
    return normalize(gold) in normalize(response)


def run_qa(ex: dict, retrieve_fn, k: int = 2) -> tuple[str, bool]:
    retrieved  = retrieve_fn(ex)[:k]
    context    = "\n\n".join(f"[{p['title']}]\n{p['text']}" for p in retrieved)
    messages   = [
        {"role": "system",
         "content": ("Answer the question using ONLY the provided Wikipedia excerpts. "
                     "Give a short, direct answer — no more than one sentence.")},
        {"role": "user",
         "content": f"Context:\n{context}\n\nQuestion: {ex['question']}"},
    ]
    response = llm.chat(messages, temperature=0, max_new_tokens=80)
    return response, answer_correct(response, ex["answer"])


N_LLM = 20  # LLM calls per system
sample = bridge_examples[:N_LLM]  # focus on bridge — where the gap is largest

print(f"Running end-to-end QA on {N_LLM} bridge questions ...")
rag_correct   = sum(run_qa(ex, rag_retrieve)[1]   for ex in sample)
grag_correct  = sum(run_qa(ex, grag_fn)[1]        for ex in sample)

print(f"\nAnswer accuracy on {N_LLM} bridge questions:")
print(f"  RAG      : {rag_correct}/{N_LLM}  ({rag_correct/N_LLM:.0%})")
print(f"  GraphRAG : {grag_correct}/{N_LLM}  ({grag_correct/N_LLM:.0%})")

### Exercise 4.2.7 `[Extension]` — Retrieval Recall@k curve

Plot Full Retrieval Recall as a function of k (from k=1 to k=10) for both RAG and GraphRAG (using `k_seed=1, hops=1`). At which k does GraphRAG saturate? What does this tell you about the density of the paragraph graph?

In [ ]:
# YOUR CODE HERE
# Hint: Plot Full Retrieval Recall as a function of k (from k=1 to k=10) for both RAG and GraphRAG (using k_seed=1, hops=1). ...
raise NotImplementedError("Complete this exercise")

---
## Summary

| System | Full Recall@2 (all) | Full Recall@2 (bridge) | Full Recall@2 (comparison) |
|---|---|---|---|
| RAG (dense) | ~55% | ~45% | ~75% |
| GraphRAG | ~65% | ~60% | ~78% |
| Published SotA (MDR, Baleen) | >90% | >90% | >90% |

*Numbers are approximate and depend on the random 300-example subset.*

**Takeaways:**
- HotpotQA is hard: even with the right 10 candidates available, dense retrieval fails ~40–50% of the time on bridge questions.
- GraphRAG consistently improves over RAG, especially for bridge questions — the graph-walk discovers paragraphs connected to the query only through intermediate entities.
- The gap to state-of-the-art (MDR, Baleen) is large — those systems do multi-step retrieval with 10–100× more compute.
- Our single-step GraphRAG is a sweet spot: one hop, one forward pass, meaningful gain.

**Next → Lab 4.3:** We train a **GCN directly on the paragraph classification task** (which paragraph is a supporting fact?) and compare its retrieval quality to RAG and GraphRAG. This is the core idea of He et al. (2024) — G-Retriever.